In [ ]:
%pip install  pandas requests

In [ ]:
import requests
import pandas as pd

In [ ]:
BASE = "https://pokeapi.co/api/v2/"

N = 151 #Toda la primera generación de pokémon

LEGENDARY = [144, 145, 146, 150, 151] #Los legendarios de la primera generación

print("Listo")

In [ ]:
r =  requests.get(f"{BASE}pokemon/25", timeout=30)
r.raise_for_status()
data = r.json()

print("name:", data["name"])
print("height:", data["height"], "| weight:", data["weight"])
print("claves:", list(data.keys())[:10])

In [ ]:
print("types:", data["types"])
print()
print("stats[0]:", data["stats"][0])

In [ ]:
def flatten (data):
    """JSON de un Pokémon -> fila plana (dict) lista para el CSV."""

    stats = { s["stat"]["name"].replace("-", "_"): s["base_stat"] for s in data["stats"] }
    types = [ t["type"]["name"] for t in data["types"]]
    sprite = (data["sprites"]["other"]["official-artwork"]["front_default"] or data["sprites"]["front_default"])

    return {
        "id": data["id"],
        "name": data["name"].capitalize(),
        "type_1": types[0],
        "type_2": types[1] if len(types) > 1 else None,
        **stats,
        "total": sum(stats.values()),
        "height_m": data["height"] / 10,
        "weight_kg": data["weight"] / 10,
        "base_exp": data.get("base_experience"),
        "legendary": data["id"] in LEGENDARY,
        "sprite": sprite,
    }

#probamos con pikachu
flatten(data)

In [ ]:
rows = []
for i in range(1, N+1):
    r =  requests.get(f"{BASE}pokemon/{i}", timeout=30)
    r.raise_for_status()
    rows.append(flatten(r.json()))
    print(f" {i}/{N} {rows[-1]['name']}", end="\r")


df = pd.DataFrame(rows)
print("\nForma:", df.shape)
df.head()

In [ ]:
df.to_csv("pokemon.csv", index=False) #columna de indice
print("Archivo pokemon.csv creado con éxito")